# Comparison and Visualisations

This notebook loads all saved experiment results, builds a master comparison
table, and generates the publication figures for the Medium article.

## 1. Setup

In [1]:
import os
import sys
import time

import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

assert os.path.exists(os.path.join('..', 'data', 'Breast_Cancer.csv')), \
    "Dataset not found. Place Breast_Cancer.csv in the data/ folder."

from src.preprocessing import load_and_clean, split_features_target, get_train_test_split
from src.feature_engineering import add_features
from src.sampling import apply_sampler
from src.models import build_model, train_model
from src.evaluation import load_results
from src import visualization

DATA_PATH = os.path.join('..', 'data', 'Breast_Cancer.csv')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
METRICS_DIR = os.path.join('..', 'results', 'metrics')
os.makedirs(FIGURES_DIR, exist_ok=True)

baseline_df = load_results(os.path.join(METRICS_DIR, 'baseline_results.csv'))
sampling_df = load_results(os.path.join(METRICS_DIR, 'sampling_results.csv'))
algo_df = load_results(os.path.join(METRICS_DIR, 'algo_results.csv'))

results_df = pd.concat([baseline_df, sampling_df, algo_df], ignore_index=True)
print(f'Combined results: {len(results_df)} models')
results_df.head()

Combined results: 11 models


,model_name,threshold,accuracy,recall_dead,precision_dead,f1_dead,f2_dead,roc_auc,pr_auc,false_negatives,recall_alive,precision_alive,f1_alive,timestamp
0,Vanilla RF,0.50,0.8373,0.1138,0.3889,0.1761,0.1326,0.7018,0.9223,109,0.9677,0.8583,0.9097,2026-06-16T12:25:00.490618
1,RF class_weight=balanced,0.50,0.8460,0.0894,0.4783,0.1507,0.1068,0.7034,0.9246,112,0.9824,0.8568,0.9153,2026-06-16T12:25:05.611918
2,RF + NONE,0.21,0.7292,0.4309,0.2637,0.3272,0.3824,0.7034,0.9246,70,0.7830,0.8841,0.8305,2026-06-16T13:20:17.331039
3,RF + SMOTE,0.21,0.6261,0.6911,0.2443,0.3609,0.5060,0.7040,0.9283,38,0.6144,0.9168,0.7357,2026-06-16T13:20:25.105668
4,RF + ADASYN,0.21,0.6037,0.6423,0.2232,0.3312,0.4669,0.6995,0.9270,44,0.5968,0.9024,0.7184,2026-06-16T13:20:32.238487


## 2. Master comparison table

In [2]:
key_cols = ['model_name', 'recall_dead', 'precision_dead', 'f1_dead', 'f2_dead',
            'false_negatives', 'pr_auc', 'roc_auc', 'threshold']
master = results_df[key_cols].sort_values('recall_dead', ascending=False).reset_index(drop=True)

styled = master.style.format({
    'recall_dead': '{:.4f}', 'precision_dead': '{:.4f}',
    'f1_dead': '{:.4f}', 'f2_dead': '{:.4f}',
    'pr_auc': '{:.4f}', 'roc_auc': '{:.4f}',
})

highlight_cols = ['recall_dead', 'pr_auc', 'roc_auc']
for col in highlight_cols:
    if col in master.columns:
        max_val = master[col].max()
        styled = styled.map(
            lambda v, c=col, m=max_val: 'font-weight: bold' if v == m else '',
            subset=[col]
        )

min_fn = master['false_negatives'].min()
styled = styled.map(
    lambda v: 'font-weight: bold' if v == min_fn else '',
    subset=['false_negatives']
)

styled

,model_name,recall_dead,precision_dead,f1_dead,f2_dead,false_negatives,pr_auc,roc_auc,threshold
0,EasyEnsemble,1.0000,0.1528,0.2651,0.4742,0,0.9456,0.7674,0.210000
1,Balanced RF,0.9675,0.1779,0.3005,0.5125,4,0.9285,0.7094,0.210000
2,RF + SMOTEENN,0.8862,0.2171,0.3488,0.5483,14,0.9344,0.7290,0.210000
3,RF + SMOTE,0.6911,0.2443,0.3609,0.5060,38,0.9283,0.7040,0.210000
4,RF + SMOTETOMEK,0.6748,0.2371,0.3510,0.4929,40,0.9256,0.7020,0.210000
5,RF + ADASYN,0.6423,0.2232,0.3312,0.4669,44,0.9270,0.6995,0.210000
6,LightGBM,0.4878,0.2091,0.2927,0.3851,63,0.9177,0.6727,0.210000
7,RF + NONE,0.4309,0.2637,0.3272,0.3824,70,0.9246,0.7034,0.210000
8,XGBoost,0.1789,0.3385,0.2340,0.1975,101,0.9195,0.6719,0.210000
9,Vanilla RF,0.1138,0.3889,0.1761,0.1326,109,0.9223,0.7018,0.500000


## 3. False negatives chart

In [3]:
visualization.plot_false_negatives_bar(
    results_df,
    save_path=os.path.join(FIGURES_DIR, '04_false_negatives_bar.png')
)
print('Saved: results/figures/04_false_negatives_bar.png')

Saved: results/figures/04_false_negatives_bar.png


## 4. Recall vs Precision scatter

In [4]:
visualization.plot_recall_precision_scatter(
    results_df,
    save_path=os.path.join(FIGURES_DIR, '05_recall_precision_scatter.png')
)
print('Saved: results/figures/05_recall_precision_scatter.png')

Saved: results/figures/05_recall_precision_scatter.png


## 5. PR curves

Re-train a representative subset of models to obtain `predict_proba` outputs for
overlay PR curves.

In [5]:
df = load_and_clean(DATA_PATH)
df = add_features(df)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = get_train_test_split(X, y)

pr_models = [
    ('Vanilla RF', 'vanilla_rf', None, X_train, y_train),
    ('RF + SMOTE', 'vanilla_rf', 'smote', X_train, y_train),
    ('Balanced RF', 'balanced_rf', None, X_train, y_train),
    ('XGBoost', 'xgboost', None, X_train, y_train),
]

pr_results = []
scale_pos_weight = (y_train == 1).sum() / (y_train == 0).sum()

for model_name, model_type, sampler, train_X, train_y in pr_models:
    print('=' * 60)
    print(f'RE-TRAINING for PR curve: {model_name}')
    print('=' * 60)
    try:
        if sampler:
            train_X, train_y = apply_sampler(train_X, train_y, method=sampler)
        if model_name == 'Vanilla RF':
            model = build_model('vanilla_rf')
        elif model_type == 'vanilla_rf':
            model = build_model('vanilla_rf', class_weight='balanced')
        elif model_type == 'xgboost':
            model = build_model('xgboost', scale_pos_weight=scale_pos_weight)
        else:
            model = build_model(model_type)

        start = time.time()
        model = train_model(model, train_X, train_y)
        print(f'Trained in {time.time() - start:.1f}s')

        y_prob = model.predict_proba(X_test)[:, 1]
        pr_results.append({
            'model_name': model_name,
            'y_true': y_test,
            'y_prob': y_prob,
        })
    except Exception as e:
        print(f'ERROR re-training {model_name}: {e}')

visualization.plot_pr_curves(
    pr_results,
    save_path=os.path.join(FIGURES_DIR, '03_pr_curves_all_models.png')
)
print('Saved: results/figures/03_pr_curves_all_models.png')

RE-TRAINING for PR curve: Vanilla RF


Trained in 4.0s


RE-TRAINING for PR curve: RF + SMOTE
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}


Resampled shape: (5452, 30)
Resampled class counts: {1: 2726, 0: 2726}


Trained in 4.1s


RE-TRAINING for PR curve: Balanced RF


Trained in 7.3s


RE-TRAINING for PR curve: XGBoost


Trained in 1.3s


Saved: results/figures/03_pr_curves_all_models.png


## 5b. Confusion matrices — top models

Confusion matrices for the three models with highest `recall_dead`, re-trained on the same split to visualise false negatives.

In [6]:
from src.evaluation import apply_threshold

top_models = results_df.sort_values('recall_dead', ascending=False).head(3)
THRESHOLD = 0.21

df = load_and_clean(DATA_PATH)
df = add_features(df)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = get_train_test_split(X, y)
scale_pos_weight = (y_train == 1).sum() / (y_train == 0).sum()

train_specs = {
    'Balanced RF': ('balanced_rf', None, X_train, y_train),
    'EasyEnsemble': ('easy_ensemble', None, X_train, y_train),
    'XGBoost': ('xgboost', None, X_train, y_train),
    'LightGBM': ('lightgbm', None, X_train, y_train),
    'RF class_weight=balanced': ('vanilla_rf', None, X_train, y_train),
    'Vanilla RF': ('vanilla_rf', None, X_train, y_train),
    'RF + SMOTE': ('vanilla_rf', 'smote', X_train, y_train),
    'RF + ADASYN': ('vanilla_rf', 'adasyn', X_train, y_train),
    'RF + SMOTEENN': ('vanilla_rf', 'smoteenn', X_train, y_train),
    'RF + SMOTETOMEK': ('vanilla_rf', 'smotetomek', X_train, y_train),
    'RF + NONE': ('vanilla_rf', 'none', X_train, y_train),
}

for idx, row in top_models.iterrows():
    model_name = row['model_name']
    threshold = float(row['threshold']) if pd.notna(row['threshold']) else THRESHOLD
    if model_name not in train_specs:
        print(f'Skipping {model_name} — no train spec')
        continue
    model_type, sampler, train_X, train_y = train_specs[model_name]
    print('=' * 60)
    print(f'CONFUSION MATRIX: {model_name}')
    print('=' * 60)
    try:
        if sampler and sampler != 'none':
            train_X, train_y = apply_sampler(train_X, train_y, method=sampler)
        if model_name == 'Vanilla RF':
            model = build_model('vanilla_rf')
        elif model_type == 'vanilla_rf':
            model = build_model('vanilla_rf', class_weight='balanced')
        elif model_type == 'xgboost':
            model = build_model('xgboost', scale_pos_weight=scale_pos_weight)
        else:
            model = build_model(model_type)
        model = train_model(model, train_X, train_y)
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = apply_threshold(y_prob, threshold=threshold)
        safe_name = model_name.lower().replace(' ', '_').replace('+', 'plus')
        visualization.plot_confusion_matrix(
            y_test, y_pred, model_name,
            save_path=os.path.join(FIGURES_DIR, f'08_confusion_{safe_name}.png')
        )
        print(f'Saved confusion matrix for {model_name}')
    except Exception as e:
        print(f'ERROR plotting confusion matrix for {model_name}: {e}')


CONFUSION MATRIX: EasyEnsemble


Saved confusion matrix for EasyEnsemble
CONFUSION MATRIX: Balanced RF


Saved confusion matrix for Balanced RF
CONFUSION MATRIX: RF + SMOTEENN
Original shape: (3219, 30)
Original class counts: {1: 2726, 0: 493}
Resampled shape: (3822, 30)
Resampled class counts: {0: 2363, 1: 1459}


Saved confusion matrix for RF + SMOTEENN


## 6. Master comparison chart

In [7]:
visualization.plot_master_comparison(
    results_df,
    save_path=os.path.join(FIGURES_DIR, '07_master_comparison.png')
)
print('Saved: results/figures/07_master_comparison.png')

Saved: results/figures/07_master_comparison.png


## 7. Key findings

Run the cell below after all notebooks have completed to auto-summarise results.

In [8]:
best_recall = results_df.loc[results_df['recall_dead'].idxmax()]
best_pr = results_df.loc[results_df['pr_auc'].idxmax()]
fewest_fn = results_df.loc[results_df['false_negatives'].idxmin()]

print('KEY FINDINGS')
print('=' * 60)
print(f"Best recall (Dead):  {best_recall['model_name']} — recall_dead={best_recall['recall_dead']:.4f}")
print(f"Best PR-AUC:         {best_pr['model_name']} — pr_auc={best_pr['pr_auc']:.4f}")
print(f"Fewest false negatives: {fewest_fn['model_name']} — FN={int(fewest_fn['false_negatives'])}")
print()
print('Trade-off: higher recall on Dead patients generally lowers precision,')
print('meaning more Alive patients may be flagged as high-risk. Threshold tuning')
print('and algorithm choice both shift this balance — neither metric alone tells')
print('the full clinical story.')

KEY FINDINGS
Best recall (Dead):  EasyEnsemble — recall_dead=1.0000
Best PR-AUC:         EasyEnsemble — pr_auc=0.9456
Fewest false negatives: EasyEnsemble — FN=0

Trade-off: higher recall on Dead patients generally lowers precision,
meaning more Alive patients may be flagged as high-risk. Threshold tuning
and algorithm choice both shift this balance — neither metric alone tells
the full clinical story.
